# Restaurant Revenue Prediction — Analyse Exploratoire

**Objectif :** Explorer et comprendre les données pour guider la modélisation.

**Dataset :** TFI Restaurant Revenue Prediction (Kaggle)

## 0. Imports & chargement des données

In [2]:
import sys
sys.path.insert(0, "../src")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from restaurant_revenue.data.load import load_data

train, test = load_data()
print(f"Train : {train.shape} | Test : {test.shape}")

Train : (137, 43) | Test : (100000, 42)


## 1. Aperçu général

In [3]:
print("=== TYPES ===")
print(train.dtypes)
print("\n=== APERÇU ===")
train.head()

=== TYPES ===
Id              int64
Open Date      object
City           object
City Group     object
Type           object
P1              int64
P2            float64
P3            float64
P4            float64
P5              int64
P6              int64
P7              int64
P8              int64
P9              int64
P10             int64
P11             int64
P12             int64
P13           float64
P14             int64
P15             int64
P16             int64
P17             int64
P18             int64
P19             int64
P20             int64
P21             int64
P22             int64
P23             int64
P24             int64
P25             int64
P26           float64
P27           float64
P28           float64
P29           float64
P30             int64
P31             int64
P32             int64
P33             int64
P34             int64
P35             int64
P36             int64
P37             int64
revenue       float64
dtype: object

=== APERÇU ===


,Id,Open Date,City,City Group,Type,P1,P2,P3,P4,P5,...,P29,P30,P31,P32,P33,P34,P35,P36,P37,revenue
0,0,07/17/1999,İstanbul,Big Cities,IL,4,5.0,4.0,4.0,2,...,3.0,5,3,4,5,5,4,3,4,5653753.0
1,1,02/14/2008,Ankara,Big Cities,FC,4,5.0,4.0,4.0,1,...,3.0,0,0,0,0,0,0,0,0,6923131.0
2,2,03/09/2013,Diyarbakır,Other,IL,2,4.0,2.0,5.0,2,...,3.0,0,0,0,0,0,0,0,0,2055379.0
3,3,02/02/2012,Tokat,Other,IL,6,4.5,6.0,6.0,4,...,7.5,25,12,10,6,18,12,12,6,2675511.0
4,4,05/09/2009,Gaziantep,Other,IL,3,4.0,3.0,4.0,2,...,3.0,5,1,3,2,3,4,3,3,4316715.0


### Observations — Structure du dataset

- **137 observations** dans le train, **100 000** dans le test : le jeu d'entraînement est **extrêmement petit**. Ce déséquilibre est un enjeu majeur du projet (risque de sur-apprentissage, validation délicate).
- **43 colonnes** : 1 cible (`revenue`), 4 catégorielles (`Open Date`, `City`, `City Group`, `Type`), 37 features numériques obfusquées (`P1`–`P37`), 1 identifiant (`Id`).
- Les features `P1`–`P37` sont anonymisées — on ne connaît pas leur sémantique, mais certaines sont des entiers (scores ?) et d'autres des flottants (ratios ?).
- `Open Date` est un `object` : il faudra la parser et en extraire une feature temporelle (ancienneté du restaurant).
- `Id` ne porte aucune information prédictive : à exclure de la modélisation.

### 1.2 Valeurs manquantes

In [4]:
def missing_report(df, name):
    miss = df.isnull().sum()
    miss = miss[miss > 0].sort_values(ascending=False)
    if miss.empty:
        print(f"{name} : aucune valeur manquante ✓")
    else:
        pct = (miss / len(df) * 100).round(2)
        print(f"{name} — {len(miss)} colonnes avec des NaN :\n")
        print(pd.DataFrame({"count": miss, "pct (%)": pct}).to_string())

missing_report(train, "Train")
print()
missing_report(test, "Test")

Train : aucune valeur manquante ✓

Test : aucune valeur manquante ✓


### 1.3 Doublons

In [5]:
n_dup_train = train.duplicated().sum()
n_dup_test  = test.duplicated().sum()
print(f"Doublons train : {n_dup_train}")
print(f"Doublons test  : {n_dup_test}")

Doublons train : 0
Doublons test  : 0


### 1.4 Statistiques descriptives

In [7]:
train.describe(include="all").T.style.format(precision=2)

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
Id,137.00,nan,nan,nan,68.00,39.69,0.00,34.00,68.00,102.00,136.00
Open Date,137,134,02/23/2010,2,nan,nan,nan,nan,nan,nan,nan
City,137,34,İstanbul,50,nan,nan,nan,nan,nan,nan,nan
City Group,137,2,Big Cities,78,nan,nan,nan,nan,nan,nan,nan
Type,137,3,FC,76,nan,nan,nan,nan,nan,nan,nan
P1,137.00,nan,nan,nan,4.01,2.91,1.00,2.00,3.00,4.00,12.00
P2,137.00,nan,nan,nan,4.41,1.51,1.00,4.00,5.00,5.00,7.50
P3,137.00,nan,nan,nan,4.32,1.03,0.00,4.00,4.00,5.00,7.50
P4,137.00,nan,nan,nan,4.37,1.02,3.00,4.00,4.00,5.00,7.50
P5,137.00,nan,nan,nan,2.01,1.21,1.00,1.00,2.00,2.00,8.00


### Synthèse — Section 1

| Aspect | Observation | Implication |
|---|---|---|
| Taille train | 137 lignes seulement | Cross-validation obligatoire (KFold, LeavePOut), modèles simples à privilégier en premier |
| Valeurs manquantes | Aucune dans train ni test | Pas de stratégie d'imputation nécessaire a priori |
| Doublons | Aucun | Dataset propre |
| Features P1–P37 | Anonymisées, mix int/float | Explorer distributions individuelles et corrélations avec revenue |
| `Open Date` | Chaîne texte | À convertir en `days_since_open` ou `years_since_open` (feature engineering) |
| `Id` | Identifiant séquentiel | À supprimer avant modélisation |
| `City` | 50+ villes distinctes | Trop de modalités pour un one-hot direct → target encoding ou regroupement `City Group` |